# Nube Fit: ¿subimos el precio?
Se estima la elasticidad de cada segmento, se busca el precio óptimo con las reglas de la empresa, se mide el efecto cruzado y se revisa el churn máximo que tolera cada precio nuevo.

## 1. Datos

In [0]:
import numpy as np, pandas as pd

D = 0.008

datos = pd.DataFrame({
    'segmento': ['Constantes', 'Casuales'],
    'precio':   [20, 12],
    'costo':    [8, 8],
    'unidades': [1000, 2000],
    'eps_x':    [0.5, 0.2],
    'churn':    [0.03, 0.06],
}).set_index('segmento')
datos

## 2. Elasticidad

In [0]:
pruebas = pd.DataFrame({
    'segmento': ['Constantes'] * 5 + ['Casuales'] * 5,
    'precio':   [18, 19, 20, 21, 22,  10.8, 11.4, 12, 12.6, 13.2],
    'unidades': [1159, 1074, 1000, 934, 875,  2603, 2274, 2000, 1770, 1576],
})
pruebas

In [0]:
g = pruebas[pruebas.segmento == 'Constantes']

eps_const = np.polyfit(np.log(g.precio), np.log(g.unidades), 1)[0]
print(f'Elasticidad de Constantes: {eps_const:.2f}')

In [0]:
g = pruebas[pruebas.segmento == 'Casuales']
eps_casu = np.polyfit(np.log(g.precio), np.log(g.unidades), 1)[0]
print(f'Elasticidad de Casuales: {eps_casu:.2f}')
datos['eps'] = [round(eps_const, 1), round(eps_casu, 1)]
datos

In [0]:
print('%ΔQ ÷ %ΔP:', round(((875 - 1000) / 1000) / ((22 - 20) / 20), 2))
print('con logaritmos:', round(np.log(875 / 1000) / np.log(22 / 20), 2))

**Respuesta.** La elasticidad es -1.4 en Constantes y -2.5 en Casuales, así que Casuales es más sensible al precio. La cuenta rápida con dos precios da -1.25 porque usa cambios porcentuales simples. La regresión usa logaritmos y los cinco precios, y da -1.40.

## 3. Precio óptimo y reglas

In [0]:
def precio_optimo(C, eps):
    return C * eps / (1 + eps)

def aplicar_reglas(p_hoy, p_opt, C):
    p = min(p_opt, p_hoy * 1.10)
    p = max(p, p_hoy * 0.90)
    p = max(p, 1.2 * C)
    return p

def utilidad(p, p_hoy, C, Q, eps):
    unidades = Q * (p / p_hoy) ** eps
    return (p - C) * unidades

def clv(m, c, d=D):
    return m * (1 - c) / (c + d)

def churn_maximo(m_nuevo, clv_hoy, d=D):
    return (m_nuevo - clv_hoy * d) / (m_nuevo + clv_hoy)

In [0]:
s = datos.loc['Constantes']

p_opt  = precio_optimo(s.costo, s.eps)
p_prop = aplicar_reglas(s.precio, p_opt, s.costo)

print(f'Precio óptimo P*:          ${p_opt:.2f}')
print(f'Precio propuesto (reglas): ${p_prop:.2f}')
print(f'Utilidad hoy:      {utilidad(s.precio, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad propuesta:{utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')

In [0]:
u_opt = utilidad(p_opt, s.precio, s.costo, s.unidades, s.eps)
print(f'Utilidad en P*: {u_opt:,.0f}')
print(f'Costo de la restricción: {u_opt - utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')

In [0]:
s = datos.loc['Casuales']
p_opt = precio_optimo(s.costo, s.eps)
p_prop = aplicar_reglas(s.precio, p_opt, s.costo)
u_opt = utilidad(p_opt, s.precio, s.costo, s.unidades, s.eps)
print(f'P*: ${p_opt:.2f} | propuesto: ${p_prop:.2f}')
print(f'Utilidad hoy: {utilidad(s.precio, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad propuesta: {utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad en P*: {u_opt:,.0f}')

**Respuesta.**
- **Constantes:** P* es $28, pero el precio solo puede subir 10%, así que se propone $22. La utilidad pasa de 12,000 a 12,251 y en P* sería 12,487. La restricción cuesta 236.
- **Casuales:** P* es $13.33 y el tope de 10% lo deja en $13.20. La utilidad sube de 8,000 a 8,195. La restricción casi no cuesta (2) porque P* está muy cerca del tope.

## 4. Efecto cruzado

In [0]:
def utilidad_total(p_const, p_casu):
    a, b = datos.loc['Constantes'], datos.loc['Casuales']
    q_const = a.unidades * (p_const / a.precio) ** a.eps * (p_casu / b.precio) ** b.eps_x
    q_casu  = b.unidades * (p_casu / b.precio) ** b.eps * (p_const / a.precio) ** a.eps_x
    u_const = (p_const - a.costo) * q_const
    u_casu  = (p_casu - b.costo) * q_casu
    return u_const, u_casu

hoy = utilidad_total(20, 12)
sube = utilidad_total(22, 12)
print(f'Constantes: {hoy[0]:,.0f} -> {sube[0]:,.0f}')
print(f'Casuales:   {hoy[1]:,.0f} -> {sube[1]:,.0f}')
print(f'Total:      {sum(hoy):,.0f} -> {sum(sube):,.0f}')

In [0]:
nuevo = utilidad_total(22, 13.2)
print(f'Constantes: {hoy[0]:,.0f} -> {nuevo[0]:,.0f}')
print(f'Casuales:   {hoy[1]:,.0f} -> {nuevo[1]:,.0f}')
print(f'Total:      {sum(hoy):,.0f} -> {sum(nuevo):,.0f}')
print(f'Valor del efecto cruzado: {sum(nuevo) - (12251 + 8195):,.0f}')

**Respuesta.** Al subir solo Constantes, su utilidad gana 251 y la de Casuales gana 390, porque parte de los clientes de Constantes se pasa a Casuales. De la ganancia total de 642, el 61% viene de Casuales. Si suben los dos segmentos, el total es 21,082, que son 636 más que la suma de las utilidades por separado (20,446). Un cliente de Constantes vale unos 306 y uno de Casuales unos 55, así que la ganancia de Casuales dura menos y pesa menos a largo plazo.

## 5. Churn máximo

In [0]:
filas = {}
for seg, p_prop in [('Constantes', 22), ('Casuales', 13.2)]:
    s = datos.loc[seg]
    clv_hoy = clv(s.precio - s.costo, s.churn)
    m_nuevo = p_prop - s.costo
    filas[seg] = {'CLV hoy': round(clv_hoy, 1), 'margen nuevo': m_nuevo,
                  "churn máximo": round(churn_maximo(m_nuevo, clv_hoy), 4)}
maximos = pd.DataFrame(filas).T
maximos

In [0]:
tabla = []
for e_c in (0.5, 1, 2.5):
    fila = {'ε(c)': e_c}
    for seg in ('Constantes', 'Casuales'):
        nuevo_c = datos.loc[seg, 'churn'] * (1 + e_c * 0.10)
        fila[f'Churn nuevo {seg}'] = f'{nuevo_c:.2%}'
        fila[f'¿Cabe? {seg}'] = 'Sí' if nuevo_c <= maximos.loc[seg, 'churn máximo'] else 'No'
    tabla.append(fila)
pd.DataFrame(tabla)

**Respuesta.** Constantes tiene menos margen de error: tolera un churn de hasta 3.61% (hoy 3%) y con ε(c) = 2.5 llega a 3.75%, que ya no cabe. Casuales tolera hasta 7.86% (hoy 6%) y cabe en los tres casos. Para la decisión, el aumento en Constantes conviene probarlo con un experimento y vigilar el churn, y el de Casuales es más seguro.